# Detección de defectos en infraestructuras (`Corrosion-bi3q3`) con Faster R-CNN v2

**Asignatura:** Visión por Computador — Grado en Ingeniería Informática / Inteligencia Artificial (USC, 2026/2027)  
**Dataset asignado:** `Francesco/corrosion-bi3q3` (Roboflow-100)  
**Detector analizado:** Faster R-CNN v2 (`fasterrcnn_resnet50_fpn_v2`, pesos `COCO_V1` de `torchvision`)

Este cuaderno reúne todos los análisis, métricas, gráficas y ejemplos cualitativos requeridos para la **exposición oral** siguiendo la estructura del guion:
1. Descripción del dataset y del detector Faster R-CNN v2.
2. Metodología y resultados del **baseline** (curvas de entrenamiento, mAP, curvas PR y tiempos de inferencia).
3. **Estrategias de mejora propuestas** (hipótesis técnicas y configuración de cada experimento).
4. **Evolución y comparativa cuantitativa** (tabla comparativa, barras, curvas PR, trade-off precisión-latencia y análisis de errores estilo TIDE).
5. **Análisis cualitativo** (detecciones correctas, fallos frecuentes y comparación visual *baseline vs mejor modelo*).
6. Preparación del **Ensemble WBF** con RetinaNet v2.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Rectangle

# Localizar raíz del proyecto tanto si el notebook se abre desde la raíz como desde notebooks/
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.dataset import CorrosionDataset, NOMBRES_CLASES
from src.evaluar import cargar_predicciones, _iou_entre_cajas

DIR_RESULTADOS = RAIZ / "results"
DIR_FIGURAS = DIR_RESULTADOS / "figuras"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

COLORES_CLASE = {
    1: "#1f77b4",  # Slippage
    2: "#d62728",  # corrosion
    3: "#ff7f0e",  # crack
}

print(f"Raíz del proyecto: {RAIZ}")
print(f"Dispositivo PyTorch: {'CUDA (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

---
## 1. Descripción del dataset (`Corrosion-bi3q3`) y del modelo Faster R-CNN v2

### Particularidades clave del dataset
- **Resolución uniforme:** todas las imágenes son de `640 × 640` píxeles (`train=840`, `validation=105`, `test=304`), con imágenes negativas sin anotaciones en las tres particiones (`37 / 5 / 13`).
- **Desequilibrio de clases y sesgo de validación:** la etiqueta `0` (`corrosion-0`) no tiene instancias y se reserva para el fondo. En `train`, `Slippage` (`548`) tiene menos de la mitad de ejemplos que `corrosion` (`1.176`) y `crack` (`1.388`). Además, **`validation` no contiene ningún ejemplo de `Slippage`**, lo que obliga a evaluar siempre la generalización por clase sobre `test`.
- **Predominio extremo de objetos pequeños y alargados:** el **85,3 %** de las cajas tienen un área inferior a `32² = 1.024 px²` (mediana `234,5 px²`), y las grietas (`crack`) presentan relaciones de aspecto muy extremas.

### Arquitectura: Faster R-CNN v2 (`ResNet-50 + FPN v2`)
- Detector *two-stage* preentrenado en MS COCO (`FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1`, **46,7 AP en COCO**).
- Primera etapa (**RPN**): genera propuestas de regiones a partir de los 5 niveles de la pirámide de características (`P2–P6`).
- Segunda etapa (**RoI Heads**): extrae características fijas con `MultiScaleRoIAlign` y clasifica + refina cajas mediante `FastRCNNPredictor` adaptado a **4 clases** (`fondo`, `Slippage`, `corrosion`, `crack`).

In [ ]:
# Análisis cuantitativo de clases, tamaños y relaciones de aspecto (motivación de mejoras)
from datasets import load_from_disk

datos_hf = load_from_disk(str(RAIZ / "data" / "corrosion-bi3q3"))

filas_resumen = []
ratios_por_clase = {nombre: [] for nombre in NOMBRES_CLASES.values()}
areas_por_clase = {nombre: [] for nombre in NOMBRES_CLASES.values()}

for particion in ("train", "validation", "test"):
    cont = Counter()
    vacias = 0
    for ej in datos_hf[particion]:
        cajas = ej["objects"]["bbox"]
        cats = ej["objects"]["category"]
        if len(cajas) == 0:
            vacias += 1
        cont.update(cats)
        if particion == "train":
            for (x, y, w, h), cat in zip(cajas, cats):
                nombre = NOMBRES_CLASES[cat]
                areas_por_clase[nombre].append(w * h)
                ratios_por_clase[nombre].append(w / max(h, 1e-6))

    filas_resumen.append({
        "Partición": particion,
        "Imágenes": len(datos_hf[particion]),
        "Sin objetos": vacias,
        "Slippage": cont.get(1, 0),
        "corrosion": cont.get(2, 0),
        "crack": cont.get(3, 0),
        "Total objetos": sum(cont.values()),
    })

df_dataset = pd.DataFrame(filas_resumen)
display(df_dataset)

# Gráfica de distribución de áreas y relaciones de aspecto por clase en train
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for nombre, areas in areas_por_clase.items():
    axes[0].hist(np.log10(np.asarray(areas) + 1), bins=30, alpha=0.55, label=f"{nombre} (med={np.median(areas):.0f} px²)")
axes[0].axvline(np.log10(32**2), color="black", linestyle="--", linewidth=1.2, label="Umbral COCO small (32²)")
axes[0].set_xlabel("log10(área en px² + 1)")
axes[0].set_ylabel("Número de cajas")
axes[0].set_title("Distribución de tamaños por clase (train)")
axes[0].legend(fontsize=8)

for nombre, ratios in ratios_por_clase.items():
    axes[1].hist(np.log2(np.asarray(ratios)), bins=30, alpha=0.55, label=nombre)
axes[1].axvline(np.log2(0.5), color="black", linestyle=":", linewidth=1.2, label="Anchors defecto (0.5 y 2.0)")
axes[1].axvline(np.log2(2.0), color="black", linestyle=":", linewidth=1.2)
axes[1].set_xlabel("log2(ancho / alto)")
axes[1].set_ylabel("Número de cajas")
axes[1].set_title("Relaciones de aspecto por clase vs anchors por defecto")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(DIR_FIGURAS / "01_eda_tamanos_aspecto.png", bbox_inches="tight")
plt.show()

---
## 2. Metodología y resultados del Baseline

### Configuración del Baseline (`configs/faster_rcnn.yaml`)
- **Modelo:** `fasterrcnn_resnet50_fpn_v2(weights="COCO_V1")`, `capas_entrenables_backbone=3` (defecto de `torchvision`), resolución interna `min_size=800, max_size=1333`, anchors por defecto (`32..512`, ratios `0.5, 1.0, 2.0`).
- **Entrenamiento:** 12 épocas, `AdamW` (`lr=1e-4`, `weight_decay=1e-4`), warmup lineal (1 época) + decaimiento coseno, `batch_size=2` con acumulación de gradientes (`4 pasos`, lote efectivo `8`), precisión mixta (`AMP`), semilla fija (`42`) y volteo horizontal aleatorio (`p=0.5`).
- **Selección de modelo:** mejor checkpoint según `mAP@[0.50:0.95]` en `validation`, evaluado finalmente sobre `test`.

In [ ]:
# Registro de experimentos: nombre legible -> carpeta en results/
EXPERIMENTOS = {
    "Baseline (Faster R-CNN v2)": {
        "dir": DIR_RESULTADOS / "faster_rcnn",
        "categoria": "Baseline",
        "hipotesis": "Fine-tuning estándar COCO_V1 (AdamW, 3 capas backbone, 800 px, anchors defecto).",
    },
    "E1: Data Augmentation": {
        "dir": DIR_RESULTADOS / "exp01_data_augmentation",
        "categoria": "Aumento de datos",
        "hipotesis": "Volteo H/V + recorte con zoom (objetos pequeños) + ColorJitter (iluminación/óxido).",
    },
    "E2: Transfer Learning": {
        "dir": DIR_RESULTADOS / "exp02_transfer_learning",
        "categoria": "Transfer learning",
        "hipotesis": "Descongelado progresivo (2 ép. congelado -> 5 capas) + LR diferenciada 0.1x en backbone.",
    },
    "E3a: SGD + MultiStepLR": {
        "dir": DIR_RESULTADOS / "exp03_sgd_multistep",
        "categoria": "Hiperparámetros",
        "hipotesis": "Comparar SGD con momentum (0.9) y decaimiento escalonado frente a AdamW+cosine.",
    },
    "E3b: Anchors pequeños + 960 px": {
        "dir": DIR_RESULTADOS / "exp04_anchors_resolucion",
        "categoria": "Hiperparámetros / Anchors",
        "hipotesis": "Anchors 16..256 con ratios [0.25..4.0] y resolución 960 px para grietas y objetos <32².",
    },
    "E4: Pérdida CIoU + CE ponderada": {
        "dir": DIR_RESULTADOS / "exp05_perdidas_ciou_ponderada",
        "categoria": "Funciones de pérdida",
        "hipotesis": "Regresión CIoU para localización precisa + pesos [1, 2, 1, 1.2] para Slippage y crack.",
    },
    "E5a: Ajuste NMS + Confianza": {
        "dir": DIR_RESULTADOS / "exp06_postprocesado_nms",
        "categoria": "Inferencia / Post-procesado",
        "hipotesis": "Ajuste en validación de box_nms_thresh y box_score_thresh sin reentrenar.",
    },
    "E5b: Test-Time Augmentation (WBF)": {
        "dir": DIR_RESULTADOS / "exp07_tta_wbf",
        "categoria": "Inferencia / TTA",
        "hipotesis": "Fusión WBF de predicciones multi-vista (original + hflip + vflip).",
    },
    "E6: Ensemble WBF": {
        "dir": DIR_RESULTADOS / "exp08_ensemble_wbf",
        "categoria": "Ensemble",
        "hipotesis": "Fusión WBF de detectores complementarios para aumentar robustez y recall.",
    },
}


def cargar_metricas_experimento(dir_exp: Path) -> dict | None:
    ruta = dir_exp / "metricas_test.json"
    if not ruta.exists():
        return None
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)


def cargar_historial_experimento(dir_exp: Path) -> list[dict] | None:
    ruta = dir_exp / "historial.json"
    if not ruta.exists():
        return None
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)


# Curvas de entrenamiento del Baseline
hist_base = cargar_historial_experimento(EXPERIMENTOS["Baseline (Faster R-CNN v2)"]["dir"])
if hist_base:
    epocas = [h["epoca"] for h in hist_base]
    loss_tot = [h["perdidas_train"]["total"] for h in hist_base]
    loss_cls = [h["perdidas_train"]["loss_classifier"] for h in hist_base]
    loss_box = [h["perdidas_train"]["loss_box_reg"] for h in hist_base]
    val_map = [h["metricas_val"]["map"] for h in hist_base]
    val_map50 = [h["metricas_val"]["map_50"] for h in hist_base]
    val_small = [h["metricas_val"]["map_small"] for h in hist_base]

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))

    axes[0].plot(epocas, loss_tot, marker="o", label="Pérdida total")
    axes[0].plot(epocas, loss_cls, marker="s", linestyle="--", label="Clasificación RoI")
    axes[0].plot(epocas, loss_box, marker="^", linestyle="--", label="Regresión cajas RoI")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Pérdida (train)")
    axes[0].set_title("Evolución de pérdidas — Baseline")
    axes[0].legend()

    axes[1].plot(epocas, val_map50, marker="o", label="mAP@0.50")
    axes[1].plot(epocas, val_map, marker="s", label="mAP@[0.50:0.95]")
    axes[1].plot(epocas, val_small, marker="^", linestyle="--", label="mAP_small")
    mejor_idx = int(np.argmax(val_map))
    axes[1].axvline(epocas[mejor_idx], color="gray", linestyle=":", label=f"Mejor época ({epocas[mejor_idx]})")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("mAP (validation)")
    axes[1].set_title("Evolución de mAP en validación — Baseline")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / "02_baseline_curvas_entrenamiento.png", bbox_inches="tight")
    plt.show()
else:
    print("Aún no se encontró results/faster_rcnn/historial.json.")

In [ ]:
# Curvas Precisión-Recall (PR) del Baseline sobre test
met_base = cargar_metricas_experimento(EXPERIMENTOS["Baseline (Faster R-CNN v2)"]["dir"])

if met_base and "test" in met_base:
    t = met_base["test"]
    print("=== Baseline en test ===")
    print(f"mAP@[0.50:0.95]: {t['map']:.4f} | mAP@0.50: {t['map_50']:.4f} | mAP@0.75: {t['map_75']:.4f}")
    print(f"mAP_small: {t['map_small']:.4f} | mAP_medium: {t['map_medium']:.4f} | mAP_large: {t['map_large']:.4f}")
    if "tiempos" in t:
        print(f"Latencia: {t['tiempos']['ms_por_imagen']:.2f} ms/img ({t['tiempos']['fps']:.2f} FPS)")

    if "curvas_pr_50" in t:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
        for ax, clave_pr, titulo in [
            (axes[0], "curvas_pr_50", "Curvas PR por clase (IoU = 0.50)"),
            (axes[1], "curvas_pr_75", "Curvas PR por clase (IoU = 0.75)"),
        ]:
            curvas = t.get(clave_pr, {})
            for id_c, nombre_c in NOMBRES_CLASES.items():
                if nombre_c in curvas and curvas[nombre_c]["recall"]:
                    c = curvas[nombre_c]
                    ax.plot(
                        c["recall"],
                        c["precision_interp"],
                        linewidth=2.0,
                        color=COLORES_CLASE[id_c],
                        label=f"{nombre_c} (AP={c['ap']:.3f}, GT={c['n_gt']})",
                    )
            ax.set_xlim(0, 1.0)
            ax.set_ylim(0, 1.05)
            ax.set_xlabel("Recall")
            ax.set_title(titulo)
            ax.legend(loc="lower left")
        axes[0].set_ylabel("Precision (interpolada 101 pts)")
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "03_baseline_curvas_pr.png", bbox_inches="tight")
        plt.show()
    else:
        print(
            "El baseline aún no tiene curvas PR guardadas. Ejecuta:
"
            "uv run python -m src.evaluar --checkpoint results/faster_rcnn/mejor_modelo.pth "
            "--particion test --guardar results/faster_rcnn/metricas_test_detallado.json "
            "--guardar-preds results/faster_rcnn/predicciones_test.json"
        )
else:
    print("Aún no se completó la evaluación en test del baseline (results/faster_rcnn/metricas_test.json).")

---
## 3. Estrategias de mejora propuestas (Guion — Sección 4)

Cada estrategia responde a una hipótesis técnica motivada por el análisis del dataset y la arquitectura Faster R-CNN v2:

| Estrategia | Categoría del guion | Hipótesis técnica y modificación | Comando reproducible |
|---|---|---|---|
| **E1** | Aumento de datos | Defectos invariantes a volteos H/V y sensibles a iluminación; recortes con zoom (`0.65–0.95`) agrandan el 85,3 % de objetos pequeños. | `uv run python -m src.entrenar --config configs/experimentos/01_data_augmentation.yaml` |
| **E2** | Transfer learning | Congelar backbone 2 épocas evita corromper filtros COCO con la cabeza nueva; luego descongelar 5 bloques con `lr_backbone = 0.1 × lr`. | `uv run python -m src.entrenar --config configs/experimentos/02_transfer_learning.yaml` |
| **E3a** | Hiperparámetros / Optimizador | Evaluar `SGD (lr=0.005, momentum=0.9)` + `MultiStepLR` clásico en R-CNN frente a `AdamW + cosine`. | `uv run python -m src.entrenar --config configs/experimentos/03_sgd_optimizacion.yaml` |
| **E3b** | Hiperparámetros / Anchors + Res. | Anchors reducidos `[[16],[32],[64],[128],[256]]`, ratios `[0.25, 0.5, 1.0, 2.0, 4.0]` (grietas finas) y `min_size=960`. | `uv run python -m src.entrenar --config configs/experimentos/04_anchors_resolucion.yaml` |
| **E4** | Funciones de pérdida | Regresión **CIoU** (optimiza solapamiento/centro/aspecto en cajas pequeñas) + CE ponderada `[1.0, 2.0, 1.0, 1.2]` para `Slippage` y `crack`. | `uv run python -m src.entrenar --config configs/experimentos/05_perdidas_ciou_ponderada.yaml` |
| **E5a** | Post-procesado (NMS/Conf.) | Barrido en validación de `box_nms_thresh` y `box_score_thresh` sin reentrenar. | `uv run python scripts/evaluar_postprocesado.py nms --checkpoint <mejor.pth>` |
| **E5b** | Test-Time Augmentation (TTA) | Fusión **Weighted Boxes Fusion (WBF)** de vistas `original + hflip + vflip` en inferencia. | `uv run python scripts/evaluar_postprocesado.py tta --checkpoint <mejor.pth>` |
| **E6** | Ensemble de modelos | Fusión **WBF** combinando las predicciones JSON de los mejores detectores (Faster R-CNN v2 + RetinaNet v2). | `uv run python scripts/evaluar_postprocesado.py ensemble --preds <p1.json> <p2.json>` |

---
## 4. Evolución y comparativa cuantitativa de resultados

La celda siguiente recopila automáticamente todos los experimentos que tengan resultados en `results/` y construye:
1. Tabla comparativa completa (`mAP@[0.50:0.95]`, `ΔmAP`, `mAP@0.50`, `mAP@0.75`, `mAP_small`, AP por clase y tiempos de inferencia).
2. Gráfica de evolución de `mAP@0.50`, `mAP@[0.50:0.95]` y `mAP_small`.
3. Curva *trade-off* precisión vs tiempo de inferencia (`ms/imagen`).
4. Comparativa de curvas PR por clase y desglose de errores estilo TIDE (*Baseline vs Mejor estrategia*).

In [ ]:
filas_comp = []
metricas_por_exp = {}

for nombre_exp, meta in EXPERIMENTOS.items():
    datos = cargar_metricas_experimento(meta["dir"])
    if datos is None:
        continue
    t = datos["test"] if "test" in datos else datos
    metricas_por_exp[nombre_exp] = t
    por_clase = t.get("map_por_clase", {})
    tiempos = t.get("tiempos", {})

    filas_comp.append({
        "Experimento": nombre_exp,
        "Categoría": meta["categoria"],
        "mAP@0.50:0.95": t.get("map", np.nan),
        "mAP@0.50": t.get("map_50", np.nan),
        "mAP@0.75": t.get("map_75", np.nan),
        "mAP_small": t.get("map_small", np.nan),
        "AP Slippage": por_clase.get("Slippage", np.nan),
        "AP corrosion": por_clase.get("corrosion", np.nan),
        "AP crack": por_clase.get("crack", np.nan),
        "ms/img": tiempos.get("ms_por_imagen", np.nan),
        "FPS": tiempos.get("fps", np.nan),
    })

if filas_comp:
    df_comp = pd.DataFrame(filas_comp)
    map_baseline = df_comp.loc[df_comp["Categoría"] == "Baseline", "mAP@0.50:0.95"]
    ref = float(map_baseline.iloc[0]) if len(map_baseline) > 0 else float(df_comp["mAP@0.50:0.95"].iloc[0])
    df_comp.insert(3, "ΔmAP vs Base", df_comp["mAP@0.50:0.95"] - ref)

    display(
        df_comp.style.format({
            "mAP@0.50:0.95": "{:.4f}",
            "ΔmAP vs Base": "{:+.4f}",
            "mAP@0.50": "{:.4f}",
            "mAP@0.75": "{:.4f}",
            "mAP_small": "{:.4f}",
            "AP Slippage": "{:.4f}",
            "AP corrosion": "{:.4f}",
            "AP crack": "{:.4f}",
            "ms/img": "{:.1f}",
            "FPS": "{:.1f}",
        })
    )

    # Gráfico comparativo de mAP por estrategia
    fig, ax = plt.subplots(figsize=(11, max(4.0, 0.55 * len(df_comp) + 1.5)))
    y = np.arange(len(df_comp))
    alto = 0.26
    ax.barh(y - alto, df_comp["mAP@0.50"], height=alto, label="mAP@0.50", color="#1f77b4")
    ax.barh(y, df_comp["mAP@0.50:0.95"], height=alto, label="mAP@[0.50:0.95]", color="#2ca02c")
    ax.barh(y + alto, df_comp["mAP_small"], height=alto, label="mAP_small", color="#ff7f0e")
    ax.set_yticks(y)
    ax.set_yticklabels(df_comp["Experimento"])
    ax.invert_yaxis()
    ax.set_xlabel("Puntuación COCO (test)")
    ax.set_title("Comparativa de estrategias de mejora — Faster R-CNN v2 (Corrosion-bi3q3)")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / "04_comparativa_estrategias.png", bbox_inches="tight")
    plt.show()

    # Gráfico trade-off precisión vs latencia (si hay tiempos registrados)
    df_lat = df_comp.dropna(subset=["ms/img"])
    if len(df_lat) >= 1:
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.scatter(df_lat["ms/img"], df_lat["mAP@0.50:0.95"], s=90, color="#1f77b4")
        for _, fila in df_lat.iterrows():
            ax.annotate(
                fila["Experimento"],
                (fila["ms/img"], fila["mAP@0.50:0.95"]),
                textcoords="offset points",
                xytext=(6, 4),
                fontsize=8,
            )
        ax.set_xlabel("Tiempo de inferencia (ms / imagen)")
        ax.set_ylabel("mAP@[0.50:0.95] (test)")
        ax.set_title("Compromiso precisión vs tiempo de inferencia")
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "05_tradeoff_latencia.png", bbox_inches="tight")
        plt.show()
else:
    print("Todavía no hay métricas de test en results/. Ejecuta primero el baseline y las estrategias.")

In [ ]:
# Análisis de errores estilo TIDE y curvas PR comparativas (Baseline vs Mejor estrategia)
if metricas_por_exp:
    nombre_base = "Baseline (Faster R-CNN v2)"
    nombre_mejor = max(metricas_por_exp, key=lambda k: metricas_por_exp[k].get("map", -1.0))
    print(f"Mejor configuración actual en test: {nombre_mejor} (mAP={metricas_por_exp[nombre_mejor]['map']:.4f})")

    # 1. Desglose de errores TIDE (thr=0.50)
    claves_err = ["TP", "Loc", "Cls", "Dup", "Bkg", "Miss"]
    colores_err = {
        "TP": "#2ca02c",
        "Loc": "#ff7f0e",
        "Cls": "#9467bd",
        "Dup": "#8c564b",
        "Bkg": "#d62728",
        "Miss": "#bcbd22",
    }
    exps_con_err = [k for k, v in metricas_por_exp.items() if "analisis_errores_05" in v]
    if exps_con_err:
        fig, ax = plt.subplots(figsize=(10.5, max(3.6, 0.55 * len(exps_con_err) + 1.2)))
        y = np.arange(len(exps_con_err))
        izq = np.zeros(len(exps_con_err))
        for clave in claves_err:
            vals = np.array([metricas_por_exp[e]["analisis_errores_05"][clave] for e in exps_con_err])
            ax.barh(y, vals, left=izq, label=clave, color=colores_err[clave])
            izq += vals
        ax.set_yticks(y)
        ax.set_yticklabels(exps_con_err)
        ax.invert_yaxis()
        ax.set_xlabel("Número de instancias (GT + FP a umbral de confianza 0.50)")
        ax.set_title("Desglose de aciertos y errores frecuentes (estilo TIDE, score >= 0.50)")
        ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.15))
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "06_analisis_errores_tide.png", bbox_inches="tight")
        plt.show()

    # 2. Curvas PR por clase: Baseline vs Mejor modelo
    if (
        nombre_base in metricas_por_exp
        and "curvas_pr_50" in metricas_por_exp[nombre_base]
        and "curvas_pr_50" in metricas_por_exp[nombre_mejor]
    ):
        fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
        for ax, (id_c, nombre_c) in zip(axes, NOMBRES_CLASES.items()):
            c_base = metricas_por_exp[nombre_base]["curvas_pr_50"][nombre_c]
            c_best = metricas_por_exp[nombre_mejor]["curvas_pr_50"][nombre_c]
            ax.plot(c_base["recall"], c_base["precision_interp"], "--", linewidth=2, label=f"Baseline (AP50={c_base['ap']:.3f})")
            if nombre_mejor != nombre_base:
                ax.plot(c_best["recall"], c_best["precision_interp"], "-", linewidth=2.2, label=f"Mejor (AP50={c_best['ap']:.3f})")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.05)
            ax.set_xlabel("Recall")
            ax.set_title(f"Clase: {nombre_c} (GT={c_base['n_gt']})")
            ax.legend(loc="lower left", fontsize=8)
        axes[0].set_ylabel("Precision (IoU=0.50)")
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "07_curvas_pr_comparativa.png", bbox_inches="tight")
        plt.show()

---
## 5. Análisis cualitativo: detecciones correctas y fallos frecuentes

Visualizamos sobre la partición de `test`:
- **Cajas verdes discontinuas:** *Ground Truth* (anotaciones reales).
- **Cajas sólidas de color:** predicciones del detector (azul = `Slippage`, rojo = `corrosion`, naranja = `crack`) con su puntuación de confianza.

Seleccionamos automáticamente ejemplos representativos de **aciertos** y **fallos típicos** (falsos negativos en grietas finas, falsos positivos en zonas oxidadas no anotadas y errores de localización).

In [ ]:
def dibujar_detecciones(ax, imagen_tensor, gt, pred, titulo="", score_thr=0.50):
    img_np = imagen_tensor.permute(1, 2, 0).numpy()
    ax.imshow(img_np)

    # Ground truth en verde discontinuo
    for caja, etq in zip(gt["boxes"].numpy(), gt["labels"].numpy()):
        x1, y1, x2, y2 = caja
        ax.add_patch(
            Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="#00e676", linewidth=1.8, linestyle="--")
        )

    # Predicciones por encima del umbral
    for caja, score, etq in zip(pred["boxes"].numpy(), pred["scores"].numpy(), pred["labels"].numpy()):
        if score < score_thr:
            continue
        x1, y1, x2, y2 = caja
        color = COLORES_CLASE.get(int(etq), "yellow")
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.8))
        ax.text(
            x1,
            max(0, y1 - 3),
            f"{NOMBRES_CLASES.get(int(etq), str(etq))} {score:.2f}",
            color="white",
            fontsize=7,
            bbox={"facecolor": color, "alpha": 0.75, "pad": 1},
        )

    ax.set_title(titulo, fontsize=9)
    ax.axis("off")


def clasificar_imagenes_cualitativo(ds, preds, score_thr=0.50):
    """Ordena imágenes de test según aciertos (TP) y tipos de fallo (Miss / Bkg)."""
    resumenes = []
    for idx in range(len(ds)):
        _, gt = ds[idx]
        pred = preds[idx]
        cajas_gt = gt["boxes"].numpy()
        etq_gt = gt["labels"].numpy()

        mask = pred["scores"].numpy() >= score_thr
        cajas_p = pred["boxes"].numpy()[mask]
        etq_p = pred["labels"].numpy()[mask]

        tp = 0
        usado = np.zeros(len(cajas_gt), dtype=bool)
        if len(cajas_gt) > 0 and len(cajas_p) > 0:
            ious = _iou_entre_cajas(cajas_p, cajas_gt)
            for i in range(len(cajas_p)):
                for j in range(len(cajas_gt)):
                    if not usado[j] and etq_p[i] == etq_gt[j] and ious[i, j] >= 0.5:
                        tp += 1
                        usado[j] = True
                        break
        fn = len(cajas_gt) - tp
        fp = len(cajas_p) - tp
        resumenes.append({"idx": idx, "n_gt": len(cajas_gt), "tp": tp, "fp": fp, "fn": fn})
    return resumenes


# Elegir el mejor experimento con predicciones_test.json disponible
ruta_preds_mejor = None
nombre_cual = None
if metricas_por_exp:
    for nombre in sorted(metricas_por_exp, key=lambda k: -metricas_por_exp[k].get("map", -1.0)):
        cand = EXPERIMENTOS[nombre]["dir"] / "predicciones_test.json"
        if cand.exists():
            ruta_preds_mejor = cand
            nombre_cual = nombre
            break

if ruta_preds_mejor is not None:
    ds_test = CorrosionDataset("test")
    preds_cual = cargar_predicciones(ruta_preds_mejor)
    stats_img = clasificar_imagenes_cualitativo(ds_test, preds_cual, score_thr=0.50)

    # Top 3 aciertos (mayor TP con 0 FN y 0 FP, o mayor ratio TP/GT)
    aciertos = sorted(
        [s for s in stats_img if s["n_gt"] >= 2],
        key=lambda s: (-s["tp"] / max(s["n_gt"], 1), s["fp"] + s["fn"], -s["tp"]),
    )[:3]

    # Top 3 fallos frecuentes (mayor número de FN o FP)
    fallos = sorted(
        [s for s in stats_img if s["n_gt"] >= 1],
        key=lambda s: -(s["fn"] + s["fp"]),
    )[:3]

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.5))
    for col, s in enumerate(aciertos):
        img, gt = ds_test[s["idx"]]
        dibujar_detecciones(
            axes[0, col],
            img,
            gt,
            preds_cual[s["idx"]],
            titulo=f"Acierto (test #{s['idx']}) | TP={s['tp']}/{s['n_gt']}, FP={s['fp']}",
        )
    for col, s in enumerate(fallos):
        img, gt = ds_test[s["idx"]]
        dibujar_detecciones(
            axes[1, col],
            img,
            gt,
            preds_cual[s["idx"]],
            titulo=f"Fallo (test #{s['idx']}) | TP={s['tp']}/{s['n_gt']}, FP={s['fp']}, FN={s['fn']}",
        )

    fig.suptitle(f"Análisis cualitativo en test — {nombre_cual} (GT=verde discontinuo)", fontsize=12)
    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / "08_analisis_cualitativo.png", bbox_inches="tight")
    plt.show()
else:
    print("No se encontró ningún archivo predicciones_test.json todavía.")

---
## 6. Ensemble Faster R-CNN v2 + RetinaNet v2 (Weighted Boxes Fusion)

Cuando el modelo **RetinaNet v2** genere sus predicciones sobre `test` con el mismo formato JSON (`src/evaluar.py --guardar-preds results/retinanet/predicciones_test.json`), se puede evaluar el ensemble final ejecutando:

```powershell
uv run python scripts/evaluar_postprocesado.py ensemble `
  --preds results/<mejor_faster_rcnn>/predicciones_test.json results/retinanet/predicciones_test.json `
  --pesos 1.0 1.0 `
  --iou-wbf 0.55 `
  --salida results/exp08_ensemble_wbf
```

Al re-ejecutar este cuaderno, la fila **E6: Ensemble WBF** se incorporará automáticamente a todas las tablas y gráficas comparativas de la Sección 4.